# Phase 07 — CIC-IDS2017 Multi-class Classification

This notebook inspects and orchestrates the reusable multi-class pipeline. The actual row-stratified run is exploratory because attack labels are concentrated in individual source files. Phase 06 binary classification remains incomplete.

## 1. Project introduction

The classifier predicts among the validated CIC-IDS2017 labels. Training and evaluation live in `src/intrusion_intelligence/multiclass/`; the notebook reads their outputs and demonstrates inference.

## 2. Dataset loading

Load the Phase 04 Parquet inventory and the Phase 05 feature contract. The inventory reads target labels and verifies all file schemas; predictive columns are loaded separately in bounded batches by the training service.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

from intrusion_intelligence.feature_engineering import FeatureContract, separate_features_target_metadata
from intrusion_intelligence.multiclass.config import load_multiclass_config
from intrusion_intelligence.multiclass.data import inspect_parquet_dataset
from intrusion_intelligence.multiclass.inference import MultiClassPredictor
from intrusion_intelligence.multiclass.service import run_multiclass_experiment

CONFIG_PATH = Path('configs/multiclass.toml')
CONFIG = load_multiclass_config(CONFIG_PATH)
CONTRACT = FeatureContract.from_file(CONFIG['dataset']['feature_schema'])
INVENTORY = inspect_parquet_dataset(
    CONFIG['dataset']['parquet_directory'], CONTRACT,
    mapping_version=CONFIG['training']['mapping_version'],
)
RUN_ID = '20261103T134128Z'
EVALUATION_DIR = CONFIG['reports']['evaluation_directory'] / RUN_ID
EXPERIMENT_DIR = CONFIG['reports']['experiment_directory'] / RUN_ID
FIGURE_DIR = CONFIG['reports']['figure_directory'] / RUN_ID
with (EXPERIMENT_DIR / 'experiment_summary.json').open(encoding='utf-8') as stream:
    EXPERIMENT = json.load(stream)
print(f'{len(INVENTORY.files)} Parquet files; {len(INVENTORY.labels):,} rows; {len(INVENTORY.taxonomy.labels)} labels')

## 3. Original attack-label inspection

Labels remain their original normalized strings. Null, blank, and labels outside the feature contract are rejected by the inventory and taxonomy validators.

In [ ]:
display(pd.DataFrame([
    {'label': label, 'class_id': class_id, 'rows': INVENTORY.class_counts[label]}
    for class_id, label in enumerate(INVENTORY.taxonomy.labels)
]))

## 4. Multi-class target taxonomy

Mapping version 1.0.0 assigns `BENIGN` identifier 0 and sorts the remaining observed labels deterministically. The inverse mapping is persisted with each model bundle.

In [ ]:
display(pd.DataFrame([
    {'class_id': class_id, 'original_label': label}
    for class_id, label in INVENTORY.taxonomy.id_to_class.items()
]))

## 5. Class distribution and imbalance analysis

These are full-dataset counts, not sample estimates. The long tail makes accuracy and weighted F1 insufficient on their own.

In [ ]:
display(pd.DataFrame(INVENTORY.distribution_records()).sort_values('count', ascending=False))
display(Image(filename=str(FIGURE_DIR / 'class_distribution_log.png')))
display(pd.read_csv(EVALUATION_DIR / 'class_distribution_by_source_file.csv').pivot_table(
    index='label', columns='source_file', values='count', fill_value=0, aggfunc='sum'
))

## 6. Feature and metadata separation

The target and provenance remain outside the predictive matrix. This small preview is only for schema inspection.

In [ ]:
preview = pd.read_parquet(
    INVENTORY.files[0],
    columns=[*CONTRACT.predictive_features, CONTRACT.target_name, '__source_file', '__record_id'],
).head(50)
SAMPLE_X, SAMPLE_Y, SAMPLE_METADATA = separate_features_target_metadata(preview, CONTRACT)
print('X:', SAMPLE_X.shape, 'y:', SAMPLE_Y.shape, 'metadata:', SAMPLE_METADATA.shape)
display(SAMPLE_METADATA.head())

## 7. Feature schema validation

The fitted preprocessing wrapper checks feature names, types, order, missing required fields, infinity, and target/metadata contamination before transforming.

In [ ]:
assert list(SAMPLE_X.columns) == list(CONTRACT.predictive_features)
assert CONTRACT.target_name not in SAMPLE_X.columns
assert not set(SAMPLE_METADATA.columns) & set(SAMPLE_X.columns)
print(f'{len(CONTRACT.predictive_features)} validated predictive features; {len(CONTRACT.metadata_fields)} metadata fields')

## 8. Group-aware data splitting

The configured splitter attempts group-disjoint train/validation/test partitions using source file as the capture group. It falls back only when class coverage makes a group split infeasible.

## 9. Class coverage and split integrity checks

Most attack labels occur in a single source file, so a source-file-disjoint three-way split cannot cover all classes. The actual run explicitly used the permitted stratified-row fallback; all eight groups overlap in every partition.

In [ ]:
with (EVALUATION_DIR / 'split_audit.json').open(encoding='utf-8') as stream:
    SPLIT_AUDIT = json.load(stream)
print(SPLIT_AUDIT['requested_strategy'], '->', SPLIT_AUDIT['used_strategy'])
print(SPLIT_AUDIT['fallback_reason'])
display(pd.DataFrame(SPLIT_AUDIT['row_counts'].items(), columns=['partition', 'rows']))
display(pd.DataFrame(SPLIT_AUDIT['groups_per_partition'].items(), columns=['partition', 'source groups']))

## 10. Preprocessing integration

Each model has its own Phase 05 fitted preprocessing artifact: Logistic Regression uses the scaled `linear` profile; Random Forest uses the unscaled `tree` profile. Fit uses only the stratified training subset.

## 11. Logistic Regression training

The actual run used an 80,000-row training-only sample. Logistic Regression reached its 160-iteration limit; the warning and the unconverged status are recorded. To start a new run, set `RUN_TRAINING = True` after selecting an unused artifact version in `configs/multiclass.toml`.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    RUN_RESULT = run_multiclass_experiment(CONFIG_PATH)
else:
    print('Using the completed run:', RUN_ID)
display(pd.read_csv(EVALUATION_DIR / 'model_comparison.csv'))

## 12. Random Forest training

Random Forest used the same split and training sample, with balanced class weights calculated from full training-partition labels. No validation or test rows were resampled.

## 13. XGBoost training, if available

XGBoost was not installed in the verified environment, so no XGBoost model or metric is reported. The comparison table records it as unavailable.

## 14. Overall model evaluation

Metrics are stored with their partition identifier. Macro metrics, weighted metrics, balanced accuracy, and one-vs-rest score metrics are reported where valid.

In [ ]:
for model_name in ('logistic_regression', 'random_forest'):
    with (EVALUATION_DIR / f'{model_name}_validation_metrics.json').open(encoding='utf-8') as stream:
        print(model_name, json.load(stream))
with (EVALUATION_DIR / 'random_forest_held_out_test_metrics.json').open(encoding='utf-8') as stream:
    print('selected model held-out test:', json.load(stream))

## 15. Per-class metric analysis

Review precision, recall, F1, support, false negatives, and correct predictions together. Undefined values and unseen training classes have explicit statuses.

In [ ]:
display(pd.read_csv(EVALUATION_DIR / 'random_forest_held_out_test_per_class.csv'))

## 16. Confusion matrix analysis

Raw counts and row-normalized percentages use the same saved taxonomy ordering.

In [ ]:
display(Image(filename=str(FIGURE_DIR / 'random_forest_held_out_test_confusion_matrix_counts.png')))
display(Image(filename=str(FIGURE_DIR / 'random_forest_held_out_test_confusion_matrix_normalized.png')))
display(Image(filename=str(FIGURE_DIR / 'random_forest_held_out_test_precision_recall_curves.png')))
display(Image(filename=str(FIGURE_DIR / 'random_forest_held_out_test_roc_curves.png')))


## 17. Minority-class investigation

Heartbleed has 1 held-out test row, Infiltration has 5, and SQL Injection has 3. Their apparent recall values are not reliable estimates. Bot, Web Attack Brute Force, and Web Attack XSS have more rows but substantially weaker recall than the dominant classes.

## 18. Frequent confusion-pair analysis

Counts identify the errors to investigate; labels are not merged based on confusion.

In [ ]:
display(pd.read_csv(EVALUATION_DIR / 'random_forest_held_out_test_confusion_pairs.csv').head(20))
with (EVALUATION_DIR / 'selected_test_error_feature_summaries.json').open(encoding='utf-8') as stream:
    display(pd.DataFrame(json.load(stream)))

## 19. Model comparison

Random Forest was selected by validation macro F1. Its held-out test result is reported after that selection; test metrics were not used to choose model configuration.

In [ ]:
display(pd.read_csv(EVALUATION_DIR / 'model_comparison.csv'))

## 20. Feature importance

Random Forest impurity importance and Logistic Regression mean absolute coefficients are descriptive and do not imply causation.

In [ ]:
display(pd.read_csv(EVALUATION_DIR / 'random_forest_feature_importance.csv').head(25))
display(Image(filename=str(FIGURE_DIR / 'random_forest_feature_importance.png')))
display(pd.read_csv(EVALUATION_DIR / 'logistic_regression_feature_importance.csv').head(25))

## 21. Held-out test evaluation

Only the validation-selected Random Forest was evaluated for final test reporting. A separate deterministic pass summarized aggregate features for the most frequent error pairs; it did not alter model selection.

## 22. Artifact persistence

Versioned bundles contain the classifier, fitted preprocessor, taxonomy, model configuration, training class counts, validation metrics, and selected-model test metrics. Existing versions are not overwritten.

In [ ]:
display(pd.DataFrame([
    {'model': name, 'artifact': path}
    for name, path in EXPERIMENT['artifacts'].items()
]))
display(pd.DataFrame(EXPERIMENT['models'])[['model', 'status', 'artifact_path']].dropna(subset=['artifact_path']))

## 23. Prediction interface demonstration

The persisted predictor validates feature-only input, applies the saved preprocessor without refitting, and returns original human-readable labels with score columns aligned to the saved class mapping.

In [ ]:
PREDICTOR = MultiClassPredictor(EXPERIMENT['artifacts']['random_forest'])
DEMO = PREDICTOR.predict(SAMPLE_X.head(5))
display(pd.DataFrame(DEMO))

## 24. Findings and limitations

The dataset contains 15 labels and a 206,645:1 majority-to-minority ratio. Source-file grouping makes full class coverage across group-disjoint partitions impossible, so reported metrics use an explicitly disclosed stratified-row fallback with all source files overlapping. These results are exploratory, not cross-capture evidence. Phase 06 binary classification remains incomplete; XGBoost is unavailable; Logistic Regression did not converge; several test classes have very small support. Phase 11 remains responsible for broader validation.